# SOFA trajectory forecasting — data preparation, model development & evaluation

**TRIPOD+AI items 1, 2, 3a–3b, 4.** This is the single, canonical notebook for the project:
it develops a model that forecasts a patient's rolling SOFA (organ-failure) score every 6 h
across ICU hours 24–48, from the PhysioNet/CinC Challenge 2012 dataset, and reports how well
that model performs on data it never trained on. It supersedes the earlier
`phase1_data.ipynb` (folded in below) and is the companion artefact to
**`SOFA_forecast_project_plan.docx`**, which holds the full problem statement (§1), decision
log (§3) and the TRIPOD+AI compliance table (§7) this notebook's headers cross-reference.

**Item 3a–3b — clinical context & target population.** Organ dysfunction in ICU patients
changes hour to hour; SOFA is normally scored once every 24 h, retrospectively. This model
gives a clinician, at any 6-hour checkpoint from hour 24 onward, an estimate of where the
patient's SOFA will be a further 6 h out — turning a retrospective score into a short-horizon
early-warning signal. Target population: adult ICU stays ≥ 48 h (PhysioNet/CinC 2012 set A,
`n = 12,000`).

**Item 4 — study objective.** Model **development** with **internal** validation (train/val
split) and a single held-out **internal test** evaluation. No external (different-hospital or
different-era) validation is available or attempted — logged as a limitation (§14 below).

**⚠ Status of this notebook.** Sections 1–10 (data through baselines) are executed and their
outputs below are real. Sections 11 onward (**model training**, results, interpretation,
artefact-saving) contain complete, ready-to-run code but are **deliberately left
unexecuted** — training is being run on a separate device. Re-running the whole notebook
top-to-bottom on that device reproduces everything, including the numbers currently missing
from the Results section.


In [1]:
import os, sys, time, pathlib
sys.path.insert(0, os.path.abspath(".."))          # project root on path

import numpy as np, pandas as pd
from scipy import stats
import joblib

from modeling import config
from modeling.data_prep import parsing, splits, targets, features, qc
from modeling import models, evaluate

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
print("horizons:", config.HORIZONS_H)
for T in config.HORIZONS_H:
    print(f"  T={T:>2}h  features t<{config.feature_cutoff_min(T)}min  "
          f"target window {tuple(x//60 for x in config.target_window_min(T))}h  "
          f"now window {tuple(x//60 for x in config.now_window_min(T))}h")


C:\Conda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


horizons: (24, 30, 36, 42)
  T=24h  features t<1440min  target window (6, 30)h  now window (0, 24)h
  T=30h  features t<1800min  target window (12, 36)h  now window (6, 30)h
  T=36h  features t<2160min  target window (18, 42)h  now window (12, 36)h
  T=42h  features t<2520min  target window (24, 48)h  now window (18, 42)h


## 1&nbsp;&nbsp;Data source & participants&nbsp;&nbsp;*(TRIPOD+AI 5a–5b, 6a–6c, 7)*

**5a — source of data.** PhysioNet/Computing in Cardiology Challenge 2012, set A: 12,000 ICU
stays, publicly available, de-identified. This project uses no other data source.
**5b — accrual dates.** Not available in the de-identified release — logged as a limitation
(no admission calendar dates, so temporal drift/seasonality cannot be assessed).
**6a — setting.** Four ICU types (coronary care, cardiac-surgery recovery, medical, surgical).
**6b — eligibility.** The dataset's own inclusion rule: adult ICU stay, length of stay ≥ 48 h
(shorter stays already excluded by PhysioNet before release). No further exclusions are
applied here — all 12,000 records are used.
**6c — treatments received.** No medication data of any kind exists in this dataset —
notably **no vasopressor/inotrope name or dose**, which is why the SOFA cardiovascular axis
is capped at 0/1 throughout this project (quantified in `SOFA_modelling_approaches.docx`).
**7 — pre-processing/QC.** `data_prep/parsing.py` applies the cleaning rules below; the audit
must reproduce `sofa_calculation.ipynb`'s section-2 counts exactly.


In [2]:
t0 = time.time()
if config.RECORDS_CLEAN.exists():
    records = joblib.load(config.RECORDS_CLEAN)
    print(f"loaded {len(records)} cleaned records from cache in {time.time()-t0:.1f}s")
else:
    records = parsing.parse_all()
    joblib.dump(records, config.RECORDS_CLEAN, compress=3)
    print(f"parsed + cached {len(records)} records in {time.time()-t0:.1f}s")

audit = qc.audit_raw()
bad = {k: (audit['counts'][k], v) for k, v in qc.DOC_REF.items() if audit['counts'][k] != v}
print("section-7 audit mismatches vs doc:", bad or "NONE -- all match")
pd.DataFrame({"computed": audit['counts'], "doc": qc.DOC_REF}).assign(
    ok=lambda d: d.computed == d.doc)


loaded 12000 cleaned records from cache in 14.2s
section-7 audit mismatches vs doc: NONE -- all match


,computed,doc,ok
empty_param_rows,2403,2403,True
MAP_eq_0,86,86,True
PaO2_eq_0,3,3,True
SaO2_eq_0,2,2,True
Weight_neg1,1001,1001,True
Weight_le20,353,353,True
Weight_gt300,1,1,True
Height_lt100,17,17,True
Height_gt250,11,11,True
MechVent_rows,91644,91644,True


In [3]:
age = pd.Series({rid: d.get("Age", np.nan) for rid, (_s, d) in records.items()})
gender = pd.Series({rid: d.get("Gender", np.nan) for rid, (_s, d) in records.items()})
icutype = pd.Series({rid: d.get("ICUType", np.nan) for rid, (_s, d) in records.items()})
print(f"Age:    median {age.median():.0f}  IQR {age.quantile(.25):.0f}-{age.quantile(.75):.0f}")
print(f"Gender: male {100*(gender==1).mean():.1f}%  female {100*(gender==0).mean():.1f}%")
print("ICUType (1 CCU / 2 CardSurg / 3 MICU / 4 SICU):",
      icutype.value_counts().sort_index().to_dict())

lt = audit['last_t_hours']
print(f"\nlast observation (h): min {lt.min():.1f}  p5 {np.percentile(lt,5):.1f}  "
      f"median {np.percentile(lt,50):.1f}  max {lt.max():.1f}")
print("records ending before 42h:", int((lt < 42).sum()), " before 36h:", int((lt < 36).sum()))


Age:    median 67  IQR 53-78
Gender: male 56.1%  female 43.8%
ICUType (1 CCU / 2 CardSurg / 3 MICU / 4 SICU): {1.0: 1769, 2.0: 2530, 3.0: 4293, 4.0: 3408}

last observation (h): min 0.0  p5 46.3  median 47.5  max 48.0
records ending before 42h: 171  before 36h: 83


## 2&nbsp;&nbsp;Outcome definition&nbsp;&nbsp;*(TRIPOD+AI 8a–8c)*

**8a — outcome & time horizon.** The outcome is the **rolling SOFA score**: for a decision
time `T ∈ {24,30,36,42} h`, the outcome is the six-axis SOFA total (and per-axis sub-scores)
over the trailing-24 h window `[T−18, T+6)h` — i.e. the score **6 h ahead** of `T`. A
secondary binary outcome, `deteriorate_24h`, is `1` when that score rises ≥ 2 points versus
the score on the last fully-observed 24 h (`SOFA_now`). Both outcomes are computed
**algorithmically** by the project's rule engine (`data_prep/scoring.py`, unchanged from
`sofa_calculation.ipynb`) — never `outcomes.csv`, which is the outcome the challenge
organisers derived with data (vasopressor doses) this project does not have, and is used only
as an external reference in evaluation, never as a training signal.
**8b–8c — assessor blinding.** Not applicable: the outcome is computed by a fixed,
version-controlled scoring function, not by a human rater.


In [4]:
t0 = time.time()
tg = targets.build_targets(records)
tg.to_csv(config.TARGETS_CSV, index=False)
print(f"targets {tg.shape} in {time.time()-t0:.1f}s   rows/record: "
      f"{tg.groupby('RecordID').size().value_counts().to_dict()}")
tg.groupby("origin_h")[["sofa_total","sofa_now","sofa_delta","deteriorate_24h",
                        "cardio_instability","target_window_empty"]].mean().round(3)


targets (48000, 28) in 4.2s   rows/record: {4: 12000}


,sofa_total,sofa_now,sofa_delta,deteriorate_24h,cardio_instability,target_window_empty
origin_h,,,,,,
24,5.275,5.747,-0.483,0.046,0.684,0.002
30,4.982,5.264,-0.294,0.038,0.653,0.002
36,4.810,4.970,-0.172,0.042,0.627,0.002
42,4.631,4.798,-0.179,0.039,0.603,0.003


In [5]:
# sanity: the T=42 target window IS Day 2 [24,48)h -> must equal notebook SOFA_D2
sc = pd.read_csv(config.SOFA_SCORES_CSV)[["RecordID","SOFA_D2"]]
chk = tg[tg.origin_h == 42].merge(sc, on="RecordID")
d = chk.sofa_total.fillna(0) - chk.SOFA_D2
r = stats.pearsonr(chk.sofa_total.fillna(0), chk.SOFA_D2)[0]
print(f"T=42 sofa_total vs sofa_scores.SOFA_D2:  r = {r:.4f}   mean|diff| = {d.abs().mean():.4f}")
assert r > 0.98

oc = pd.read_csv(config.OUTCOMES_CSV)[["RecordID","SOFA"]]
ref = tg[tg.origin_h == 24].merge(oc, on="RecordID"); ref = ref[ref.SOFA != -1]
b = (ref.sofa_now - ref.SOFA).mean()  # sofa_now at T=24 == SOFA_D1
rr = stats.pearsonr(ref.sofa_now, ref.SOFA)[0]
print(f"[0,24)h rule SOFA (sofa_now @ T=24) vs outcomes.SOFA (reference only, never a feature): "
      f"r={rr:.3f} bias={b:+.3f}  (expect r 0.933, bias -0.862)")


T=42 sofa_total vs sofa_scores.SOFA_D2:  r = 1.0000   mean|diff| = 0.0000
[0,24)h rule SOFA (sofa_now @ T=24) vs outcomes.SOFA (reference only, never a feature): r=0.933 bias=-0.862  (expect r 0.933, bias -0.862)


## 3&nbsp;&nbsp;Predictors&nbsp;&nbsp;*(TRIPOD+AI 9a–9c)*

**9a — choice of predictors.** All time-varying vitals/labs in `icu_records` (36 variables:
GCS, HR, blood pressures, temperature, urine, chemistry panel, blood gases, MechVent,
bilirubin, etc.), summarised over `[0, T)` (last/first/min/max/mean/std/count/hours-since-last/
hours-of-coverage/12 h-slope), plus engineered hemodynamic/neuro/renal/respiratory signals,
the rule-based SOFA sub-scores computed from our own data on `[0,24)h` and `[T−24,T)h`, and
admission descriptors. No predictor pre-selection beyond excluding `outcomes.csv` entirely
(D6). **9b — measurement.** All predictors are derived only from measurements timestamped
`< T` — never the target window `[T,T+6)` (verified below). **9c** — not applicable; no
subjectively-assessed predictor.


In [6]:
t0 = time.time()
if config.FEATURES_CSV.exists():
    ft = pd.read_csv(config.FEATURES_CSV)
    print(f"loaded features {ft.shape} from cache in {time.time()-t0:.1f}s")
else:
    ft = features.build_features(records)
    ft.to_csv(config.FEATURES_CSV, index=False)
    print(f"built + cached features {ft.shape} in {time.time()-t0:.0f}s")
fam = pd.Series([c.split('_')[0] for c in ft.columns if c not in ('RecordID','origin_h')]).value_counts()
print(f"{ft.shape[1]-2} feature columns; families:", fam.to_dict())


loaded features (48000, 409) from cache in 1.9s
407 feature columns; families: {'v': 360, 'e': 14, 'sofa0': 14, 'sofa': 14, 'd': 5}


## 4&nbsp;&nbsp;Sample size&nbsp;&nbsp;*(TRIPOD+AI 10)*

**Item 10 — how the study size was arrived at.** This is a **secondary analysis of a fixed
public dataset**, not a prospectively powered study: the sample size (12,000 stays × 4
decision times = 48,000 (record × horizon) rows) is simply everything the dataset provides.
No formal power calculation was performed; the number of usable outcome events per axis per
horizon is reported below (Section 8) so readers can judge estimate precision themselves —
particularly for the rare `deteriorate_24h` outcome (~4 % positive).


## 5&nbsp;&nbsp;Missing data&nbsp;&nbsp;*(TRIPOD+AI 11)*

Two independent missingness mechanisms are handled differently, by design:

* **Predictor missingness** (a vital/lab never measured in `[0,T)`, e.g. no arterial blood gas)
  is left as `NaN` and passed straight to `HistGradientBoostingRegressor`/`Classifier`, which
  route missing values natively during tree splitting — no mean/median imputation, so a
  "never measured" case is not silently treated as "measured normal". Two exceptions, applied
  during feature/target construction (not here): bilirubin uses last-observation-carried-
  forward across windows (clinically not re-tested daily), and admission weight is imputed
  from the cohort's sex-specific median only for the urine-rate *feature* (flagged with
  `e_weight_imputed`).
* **Outcome missingness** (target window has no measurement for that axis) is **excluded**
  from that axis's training/evaluation rows (D17) — never zero-filled — so the model is never
  taught that "unmeasured" means "score 0". (`sofa_total`/`sofa_delta`/`deteriorate_24h`
  reported in Section 2 above *do* zero-fill missing axes, matching `sofa_calculation.ipynb`'s
  lenient convention — that is a reporting convention for the aggregate score, not a
  training target.)


In [7]:
nan_frac = ft.drop(columns=["RecordID","origin_h"]).isna().mean().sort_values()
print(f"fully-populated feature columns: {(nan_frac==0).sum()} / {len(nan_frac)}   "
      f"median NaN fraction: {nan_frac.median():.2f}")
print("10 most-missing features:")
nan_frac.tail(10)


fully-populated feature columns: 60 / 407   median NaN fraction: 0.18
10 most-missing features:


v_TroponinI_mean          0.955792
v_TroponinI_max           0.955792
v_Bilirubin_slope12h      0.984458
v_AST_slope12h            0.985000
v_ALT_slope12h            0.985042
v_TroponinT_slope12h      0.985625
v_ALP_slope12h            0.985625
v_Albumin_slope12h        0.993417
v_TroponinI_slope12h      0.997042
v_Cholesterol_slope12h    0.999958
dtype: float64

## 6&nbsp;&nbsp;Data partitioning&nbsp;&nbsp;*(TRIPOD+AI 12a)*

Record-level (a stay's four horizon-rows never straddle a split boundary), stratified on a
coarse SOFA-severity band × in-hospital death (`SOFA == -1` records form their own band),
70 / 15 / 15 train / validation / test, `random_state = 42`, **frozen**. No patient linkage
exists in this dataset, so record-level is the finest grouping available (item 16: no
separate development/evaluation *sources* exist — both come from one dataset via this
random split).


In [8]:
sp = splits.make_splits()
sp.to_csv(config.SPLITS_CSV, index=False)
print(sp.split.value_counts().to_dict())
splits.split_summary(sp)


{'train': 8400, 'val': 1800, 'test': 1800}


,n,pct,sofa_mean,sofa_p25,sofa_p75,death_rate,sofa_missing,icu1,icu2,icu3,icu4
train,8400.0,70.0,6.66,3.0,9.0,0.142,282.0,0.146,0.211,0.358,0.285
val,1800.0,15.0,6.66,3.0,9.0,0.143,61.0,0.155,0.210,0.353,0.282
test,1800.0,15.0,6.67,3.0,9.0,0.142,60.0,0.148,0.210,0.362,0.280


## 7&nbsp;&nbsp;Internal-consistency & leakage checks

Not itself a TRIPOD+AI item, but the evidence for items 7/9b/12a being satisfied in practice.


In [9]:
J = ft.merge(tg, on=["RecordID","origin_h"]).merge(sp[["RecordID","split"]], on="RecordID")
print("joined:", J.shape)
print(J.groupby(["split","origin_h"]).size().unstack(), "\n")

ok = True
for v in ["v_HR_count","v_GCS_count","v_Urine_count","v_Creatinine_count"]:
    piv = ft.pivot_table(index="RecordID", columns="origin_h", values=v)
    mono = ((piv[24]<=piv[30]) & (piv[30]<=piv[36]) & (piv[36]<=piv[42])).mean()
    ok &= mono == 1.0
    print(f"  {v:20} monotone non-decreasing in T: {mono:.4f}")
assert ok

s024 = ft.pivot_table(index="RecordID", columns="origin_h", values="sofa0_24_total")
assert (s024.nunique(axis=1) == 1).all()
print("  sofa0_24_total identical across all 4 horizons: True")

leak = [c for c in ft.columns if c.lower() in
        ("sofa","saps-i","survival","length_of_stay","in-hospital_death")]
assert not leak
print("  outcomes.csv columns in features:", leak or "none")
print("  non-empty target rows all have >=1 organ scored:",
      not ((J.target_window_empty == 0) & (J.n_present_target == 0)).any())


joined: (48000, 436)
origin_h    24    30    36    42
split                           
test      1800  1800  1800  1800
train     8400  8400  8400  8400
val       1800  1800  1800  1800 

  v_HR_count           monotone non-decreasing in T: 1.0000
  v_GCS_count          monotone non-decreasing in T: 1.0000
  v_Urine_count        monotone non-decreasing in T: 1.0000
  v_Creatinine_count   monotone non-decreasing in T: 1.0000
  sofa0_24_total identical across all 4 horizons: True
  outcomes.csv columns in features: none
  non-empty target rows all have >=1 organ scored: True


## 8&nbsp;&nbsp;Exploratory analysis & baseline models&nbsp;&nbsp;*(TRIPOD+AI 12e, partial)*

Before fitting anything, establish the bar. **Persistence**: assume no change, i.e.
`SOFA_hat(T+6) = SOFA_now`. **Last-value-per-axis**: the per-axis analogue. Any trained model
that cannot beat persistence on MAE is not adding forecast value — it would just be relearning
"the future looks like the present" (also Section 4's sample-size note: event/row counts per
horizon, needed to read the CIs below).


In [10]:
base = J.dropna(subset=["sofa_total"]).copy()
print("rows usable for the TOTAL-SOFA outcome, per split x horizon:")
print(base.groupby(["split","origin_h"]).size().unstack(), "\n")
print("deteriorate_24h positive-rate per split x horizon:")
print(base.groupby(["split","origin_h"]).deteriorate_24h.mean().unstack().round(3))


rows usable for the TOTAL-SOFA outcome, per split x horizon:
origin_h    24    30    36    42
split                           
test      1797  1797  1796  1797
train     8383  8381  8380  8377
val       1795  1794  1795  1795 

deteriorate_24h positive-rate per split x horizon:
origin_h     24     30     36     42
split                               
test      0.057  0.036  0.046  0.042
train     0.045  0.039  0.041  0.038
val       0.038  0.033  0.046  0.037


In [11]:
persist = evaluate.regression_report(
    base[base.split=="train"], y_col="sofa_total", pred_cols={"persistence": "sofa_now"})
print("TOTAL-SOFA persistence baseline (train split):")
persist


TOTAL-SOFA persistence baseline (train split):


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,8383,0.933914,0.903853,0.964094,1.662901,1.616964,1.708762
1,30,persistence,8381,0.692519,0.668649,0.717817,1.274983,1.234954,1.317147
2,36,persistence,8380,0.633890,0.612885,0.655728,1.208166,1.171200,1.244753
3,42,persistence,8377,0.599379,0.577408,0.622072,1.232203,1.189359,1.277813


In [12]:
rows = []
for axis in models.AXES:
    d = base[base.split=="train"].dropna(subset=[f"{axis}_score", f"sofa_now_{axis}"])
    r = evaluate.bootstrap_ci(d[f"{axis}_score"], d[f"sofa_now_{axis}"], evaluate._mae, n_boot=500)
    rows.append({"axis": axis, "n": r["n"], "persistence_mae": r["point"],
                "mae_lo": r["ci_lo"], "mae_hi": r["ci_hi"]})
print("per-axis persistence baseline (train split):")
pd.DataFrame(rows)


per-axis persistence baseline (train split):


,axis,n,persistence_mae,mae_lo,mae_hi
0,resp,15248,0.232096,0.224485,0.240131
1,coag,31180,0.078384,0.075369,0.081751
2,liver,13508,0.033758,0.030610,0.036793
3,cardio,33007,0.082104,0.079283,0.084998
4,neuro,32986,0.299551,0.291971,0.306995
5,renal,33095,0.081916,0.078333,0.085817


## 9&nbsp;&nbsp;Model development methods&nbsp;&nbsp;*(TRIPOD+AI 12b–12g, 13, 14, 15)*

**12b — how predictors are handled.** Raw + engineered numeric features and `origin_h` (a
small integer, treated as an ordinary numeric feature so one model can learn horizon-dependent
effects) are passed directly to gradient-boosted trees; no scaling/encoding is needed for a
tree-based model.

**12c — model type, rationale, build steps, hyperparameter tuning.** Six
`HistGradientBoostingRegressor`s (one per organ axis, **pooled across all four horizons** —
decision D12: this uses 4× more rows per model than training 24 separate axis×horizon models
would, and `origin_h` already lets a single model express horizon-specific behaviour) predict
the axis **delta** (target-window score − `SOFA_now` for that axis; D17: trained only on rows
where that axis was actually measured, never the lenient zero-fill). `HistGradientBoosting`
was chosen for native missing-value handling on this sparse data, with no extra dependency
(D8). Hyperparameters are chosen from the **fixed grid below**, each point fit on `train` and
scored on the **external** validation split (D13) — not HistGB's internal `early_stopping`,
which would silently carve its own validation rows out of `train`.

**12d — heterogeneity of performance.** Every performance number reported here carries a
bootstrap confidence interval (`evaluate.bootstrap_ci`) and a subgroup breakdown (Section 12).

**12e — performance measures.** MAE/RMSE (regression, per horizon and per axis) and
AUROC/AUPRC/Brier + calibration (deterioration classifier).

**12f — model updating.** Not applicable — first model version, no updating cycle yet.

**12g — how predictions are calculated.** `models.predict_axis_scores()`: absolute sub-score
= `sofa_now_<axis>` (a feature, computed from our data) + predicted delta, clipped to `[0,4]`;
total = sum of the six.

**13 — class imbalance.** The deterioration classifier uses `class_weight="balanced"`
(`HistGradientBoostingClassifier`, sklearn ≥ 1.6) because positives are ~4 % of rows; model
selection uses validation **AUPRC**, not AUROC, for the same reason.

**14 — fairness.** Partial: Section 12 reports performance stratified by ICU type and by
hemodynamic-instability status (a pressor-likelihood proxy); no formal fairness-metric
adjustment by age or sex is applied (Section 14/Discussion: flagged as future work, not
because it was considered unimportant but because this dataset's small size once stratified
by both age band and sex would make such estimates unstable).

**15 — model output.** Six ordinal sub-scores (0–4 each) summed to a total SOFA (0–24) six
6 h ahead of the decision time, the signed delta versus the current SOFA, and a probability
of ≥ 2-point deterioration within 24 h.


In [13]:
print("regressor grid (D13) -- 4 configurations, selected by validation MAE:")
display_grid_r = pd.DataFrame(models.REGRESSOR_GRID)
print(display_grid_r)
print("\nclassifier grid (D14) -- 4 configurations, selected by validation AUPRC:")
display_grid_c = pd.DataFrame(models.CLASSIFIER_GRID)
print(display_grid_c)
display_grid_r


regressor grid (D13) -- 4 configurations, selected by validation MAE:
   learning_rate  max_leaf_nodes  max_iter  l2_regularization  random_state
0           0.05              31       300                0.0            42
1           0.05              63       300                0.0            42
2           0.10              31       300                0.0            42
3           0.10              63       300                0.0            42

classifier grid (D14) -- 4 configurations, selected by validation AUPRC:
   learning_rate  max_leaf_nodes  max_iter  l2_regularization class_weight  random_state
0           0.05              31       300                0.0     balanced            42
1           0.05              63       300                0.0     balanced            42
2           0.10              31       300                0.0     balanced            42
3           0.10              63       300                0.0     balanced            42


,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state
0,0.05,31,300,0.0,42
1,0.05,63,300,0.0,42
2,0.10,31,300,0.0,42
3,0.10,63,300,0.0,42


## 10&nbsp;&nbsp;Model training&nbsp;&nbsp;*(TRIPOD+AI 12c)*  — ⚠ NOT EXECUTED HERE

**This cell is intentionally not run in this environment.** It fits the 6 axis regressors
(`models.train_axis_models`, ~4 configs × 6 axes = 24 fits) and the deterioration classifier
(`models.train_deterioration_model`, 4 more fits) on `train`, selecting each by `val`
performance, exactly as described in Section 9. Run this notebook top-to-bottom on the
training device to populate it and everything downstream.


In [ ]:
axis_models, axis_params, axis_grids = models.train_axis_models(ft, tg, sp)
for axis in models.AXES:
    print(f"{axis:7} chosen params: {axis_params[axis]}")
    display(axis_grids[axis])

det_model, det_params, det_grid = models.train_deterioration_model(ft, tg, sp)
print("deterioration classifier chosen params:", det_params)
display(det_grid)


## 11&nbsp;&nbsp;Results&nbsp;&nbsp;*(TRIPOD+AI 20a–20c, 21, 22, 23a, 23b)*  — ⚠ NOT EXECUTED HERE

**20a–20c.** Participant flow: all 12,000 records enter Section 1 and none are excluded
(the dataset's own ≥48 h rule is the only eligibility filter, applied before release); per-
split characteristics were shown in Section 6; a development-vs-evaluation predictor-
distribution comparison is produced below. **21.** N and outcome-event counts per split ×
horizon were shown in Section 8. **22.** The full model specification (chosen hyperparameters
+ the feature list) is printed below so the model can be re-fit exactly. **23a–23b.**
Performance with bootstrap CIs, on **validation** (used for iteration) and on **test**
(unlocked once, after model selection is final).


In [ ]:
val_pred  = models.predict_axis_scores(axis_models, ft[ft.origin_h.isin(config.HORIZONS_H)])
val_pred  = val_pred.merge(sp[["RecordID","split"]], on="RecordID")
full = val_pred.merge(tg, on=["RecordID","origin_h"])

for split_name in ("val", "test"):
    d = full[full.split == split_name].dropna(subset=["sofa_total"])
    print(f"--- {split_name} : TOTAL SOFA, model vs persistence ---")
    display(evaluate.regression_report(d, "sofa_total",
                                       {"persistence": "sofa_now", "model": "sofa_total_pred"}))
    for axis in models.AXES:
        dd = d.dropna(subset=[f"{axis}_score"])
        print(f"  {axis}:")
        display(evaluate.regression_report(dd, f"{axis}_score",
                                           {"persistence": f"sofa_now_{axis}",
                                            "model": f"{axis}_pred"}, group_col="origin_h"))

det_pred = models.predict_deterioration(det_model, ft).merge(sp[["RecordID","split"]], on="RecordID")
det_full = det_pred.merge(tg[["RecordID","origin_h","deteriorate_24h"]], on=["RecordID","origin_h"]).dropna()
for split_name in ("val", "test"):
    d = det_full[det_full.split == split_name]
    print(f"--- {split_name} : deterioration classifier ---")
    print(evaluate.classification_report_ci(d.deteriorate_24h, d.deteriorate_proba))
    display(evaluate.calibration_table(d.deteriorate_24h, d.deteriorate_proba))

print("subgroup MAE (val, total SOFA) by hemodynamic-instability status:")
display(evaluate.subgroup_report(full[full.split=="val"].dropna(subset=["sofa_total"]),
                                 "sofa_total", "sofa_total_pred", "cardio_instability"))


## 12&nbsp;&nbsp;Interpretation — permutation importance&nbsp;&nbsp;(D16)  — ⚠ NOT EXECUTED HERE

`HistGradientBoosting` has no built-in `feature_importances_`; permutation importance is
sklearn's documented alternative. Computed with `n_repeats=3` on a fixed random 2,000-row
subsample of validation (not the full ~33,600-row pooled set) for tractability — a scoping
choice (D16), not a silent shortcut.


In [ ]:
cols = models.feature_columns(ft)
for axis, model in axis_models.items():
    d = full[full.split=="val"].dropna(subset=[f"{axis}_score"])
    imp = evaluate.permutation_report(model, d[cols], d[f"{axis}_delta"])
    print(f"top features for '{axis}':")
    display(imp)

d = det_full[det_full.split=="val"]
Xd = ft.merge(sp[["RecordID","split"]], on="RecordID")
Xd = Xd[Xd.split=="val"].set_index(["RecordID","origin_h"]).loc[
    d.set_index(["RecordID","origin_h"]).index].reset_index()
imp_det = evaluate.permutation_report(det_model, Xd[cols], d.deteriorate_24h.to_numpy(),
                                      scoring="average_precision")
print("top features for the deterioration classifier:")
display(imp_det)


## 13&nbsp;&nbsp;Discussion&nbsp;&nbsp;*(TRIPOD+AI 25, 26, 27a–27c)*

**25 — overall interpretation.** *(to be completed with the Section 11 numbers once trained;*
*the qualitative points below hold regardless of the exact figures.)* A useful forecaster
must beat the persistence baseline on the **delta**, not just the absolute score — the
absolute score is dominated by current severity (Section 8). Fairness (item 25): performance
is expected to be weakest for the same patients the underlying SOFA score is weakest for —
those on vasopressors whose cardiovascular sub-score is capped at 0/1 (see Section 9's
fairness note and `SOFA_modelling_approaches.docx`).

**26 — limitations.**
- The cardiovascular axis cannot exceed 1 anywhere in this project (no vasopressor data);
  every downstream number inherits that ceiling on both the feature and target side.
- 18 of the 24 target-window hours are already observed at prediction time, so all reported
  MAEs include a large "nothing changed" component; only the delta figures isolate genuine
  forecast skill.
- No accrual dates exist in the de-identified data, so temporal drift cannot be assessed.
- Single-centre-era, single-dataset internal validation only — no external validation.
- `deteriorate_24h` is rare (~4 %); classifier performance estimates are comparatively
  imprecise (see its CI width).

**27a — handling poor-quality/unavailable input at use time.** If a required vital has no
recent measurement, `HistGradientBoosting` routes it through its native missing-value branch
rather than substituting a value — a user of this model should still be told which inputs
were absent for a given prediction (the `*_present` / `*_count` features already carry that
signal and should be surfaced, not just consumed silently).
**27b — user interaction required.** None beyond ensuring the input record covers `[0,T)`
with the same variable definitions used here; this is a decision-support estimate, not an
autonomous action.
**27c — next steps.** External validation on a different ICU/era; incorporating real
vasopressor data if it ever becomes available for this cohort; a fairness audit stratified by
age and sex once a larger cohort permits stable subgroup estimates; comparing the pooled
six-axis design (D12) against the rejected 24-separate-model alternative now that a trained
baseline exists to compare against.

**Open Science — item 18e–18f.** Data: PhysioNet/CinC Challenge 2012 (public,
physionet.org). Code: this repository (`github.com/398-21/healthcare-ai-assin`), including
`modeling/data_prep/`, `modeling/models.py`, `modeling/evaluate.py` and this notebook.


## 14&nbsp;&nbsp;Save artefacts&nbsp;&nbsp;*(TRIPOD+AI 22)*  — ⚠ NOT EXECUTED HERE


In [ ]:
out_models = config.OUT_DIR / "models"
out_models.mkdir(exist_ok=True)
for axis, m in axis_models.items():
    joblib.dump(m, out_models / f"axis_{axis}.joblib")
joblib.dump(det_model, out_models / "deteriorate_24h.joblib")

spec = {"axis_params": axis_params, "deterioration_params": det_params,
       "feature_columns": models.feature_columns(ft), "seed": config.SEED,
       "horizons": list(config.HORIZONS_H)}
import json as _json
(config.OUT_DIR / "model_spec.json").write_text(_json.dumps(spec, indent=2))
print("saved 7 models to", out_models, "and model_spec.json")
